# Person Identification — With or Without Mask

Trains a MobileNetV2-based model to recognize **who** a person is regardless of whether they are wearing a mask.  
Dataset: AFDB — 442 people, each with both unmasked and masked images.

## 1. Imports

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.20.0


## 2. Load Dataset

In [3]:
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
face_dir    = os.path.join(BASE_DIR, 'data', 'AFDB_face_dataset')
masked_dir  = os.path.join(BASE_DIR, 'data', 'AFDB_masked_face_dataset')

# Only use people who appear in BOTH datasets
face_people   = set(os.listdir(face_dir))
masked_people = set(os.listdir(masked_dir))
people = sorted(face_people & masked_people)

print(f'People in unmasked dataset : {len(face_people)}')
print(f'People in masked dataset   : {len(masked_people)}')
print(f'People in both (usable)    : {len(people)}')

People in unmasked dataset : 461
People in masked dataset   : 525
People in both (usable)    : 442


In [4]:
IMG_SIZE   = 128
MIN_MASKED = 3   # minimum masked images required per person

data   = []
labels = []
skipped = 0

for person in people:
    masked_imgs = os.listdir(os.path.join(masked_dir, person))
    masked_imgs = [f for f in masked_imgs if f.lower().endswith(('.jpg','.jpeg','.png'))]

    if len(masked_imgs) < MIN_MASKED:
        skipped += 1
        continue

    face_imgs = os.listdir(os.path.join(face_dir, person))
    face_imgs = [f for f in face_imgs if f.lower().endswith(('.jpg','.jpeg','.png'))]

    for fname in face_imgs:
        try:
            img = Image.open(os.path.join(face_dir, person, fname))
            img = img.resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
            data.append(np.array(img))
            labels.append(person)
        except:
            pass

    for fname in masked_imgs:
        try:
            img = Image.open(os.path.join(masked_dir, person, fname))
            img = img.resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
            data.append(np.array(img))
            labels.append(person)
        except:
            pass

print(f'Skipped {skipped} people with fewer than {MIN_MASKED} masked images.')
print(f'Total images loaded : {len(data)}')
print(f'Total people        : {len(set(labels))}')

Skipped 232 people with fewer than 3 masked images.
Total images loaded : 45079
Total people        : 210


## 3. Prepare Data

In [5]:
X = np.array(data, dtype='float32') / 255.0

le = LabelEncoder()
Y = le.fit_transform(labels)

num_classes = len(le.classes_)
print(f'X shape      : {X.shape}')
print(f'Num classes  : {num_classes}')
print(f'Sample names : {list(le.classes_[:5])}')

X shape      : (45079, 128, 128, 3)
Num classes  : 210
Sample names : [np.str_('baijingting'), np.str_('baojianfeng'), np.str_('caizhuoyan'), np.str_('caobingkun'), np.str_('chenbailin')]


In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

Train: (36063, 128, 128, 3)  |  Test: (9016, 128, 128, 3)


## 4. Build Model — MobileNetV2 + Identity Head

MobileNetV2 is pretrained on ImageNet and learns general visual features.  
We fine-tune it on our dataset so it learns to recognize each person with or without a mask.

In [7]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
# Freeze base initially
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(256, activation='relu')(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 210)            │        53,970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,639,890 (10.07 MB)

 Trainable params: 381,906 (1.46 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. Train — Phase 1 (frozen base, 5 epochs)

In [8]:
history1 = model.fit(
    X_train, Y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32
)

Epoch 1/5
1015/1015 ━━━━━━━━━━━━━━━━━━━━ 55s 52ms/step - accuracy: 0.0100 - loss: 5.2816 - val_accuracy: 0.0299 - val_loss: 4.7986
Epoch 2/5
1015/1015 ━━━━━━━━━━━━━━━━━━━━ 52s 51ms/step - accuracy: 0.0239 - loss: 4.8737 - val_accuracy: 0.0396 - val_loss: 4.6443
Epoch 3/5
1015/1015 ━━━━━━━━━━━━━━━━━━━━ 52s 51ms/step - accuracy: 0.0295 - loss: 4.7613 - val_accuracy: 0.0474 - val_loss: 4.5642
Epoch 4/5
1015/1015 ━━━━━━━━━━━━━━━━━━━━ 63s 62ms/step - accuracy: 0.0342 - loss: 4.6947 - val_accuracy: 0.0513 - val_loss: 4.5490
Epoch 5/5
1015/1015 ━━━━━━━━━━━━━━━━━━━━ 76s 75ms/step - accuracy: 0.0382 - loss: 4.6511 - val_accuracy: 0.0577 - val_loss: 4.5284


## 6. Fine-tune — Phase 2 (unfreeze top layers, 5 more epochs)

In [ ]:
# Unfreeze the top 30 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    X_train, Y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32
)

Epoch 1/5
 111/1015 ━━━━━━━━━━━━━━━━━━━━ 2:00:35 8s/step - accuracy: 0.0132 - loss: 5.7495

## 7. Evaluate

In [ ]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Combine both training phases for plotting
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss_h  = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc,     label='Train Accuracy')
plt.plot(epochs, val_acc, label='Val Accuracy')
plt.axvline(x=5.5, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss_h,  label='Train Loss')
plt.plot(epochs, val_loss,label='Val Loss')
plt.axvline(x=5.5, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
model.save('person_identification_model.h5')
np.save('person_label_classes.npy', le.classes_)
print('Model and label classes saved.')

## 9. Identify a Person from an Image

Detects the face, then predicts **who** the person is — masked or not.  
Shows the top 3 predictions with confidence scores.

In [ ]:
import urllib.request

cascade_path = 'haarcascade_frontalface_default.xml'
if not os.path.exists(cascade_path):
    url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'
    urllib.request.urlretrieve(url, cascade_path)
    print('Downloaded face cascade.')

face_cascade = cv2.CascadeClassifier(cascade_path)
print('Face detector ready.')

In [ ]:
def identify_person(image_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f'Could not read: {image_path}')
        print('Use the full path, e.g.:')
        print(f'  {BASE_DIR}/data/AFDB_face_dataset/yangmi/0_0_yangmi_0001.jpg')
        return

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    result  = img_rgb.copy()

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    if len(faces) == 0:
        print('No face detected — classifying full image.')
        faces = [(0, 0, img_rgb.shape[1], img_rgb.shape[0])]

    for i, (x, y, w, h) in enumerate(faces):
        pad = 10
        x1 = max(0, x - pad);  y1 = max(0, y - pad)
        x2 = min(img_rgb.shape[1], x + w + pad)
        y2 = min(img_rgb.shape[0], y + h + pad)

        face_crop = img_rgb[y1:y2, x1:x2]
        face_in   = np.array(Image.fromarray(face_crop).resize((IMG_SIZE, IMG_SIZE)), dtype='float32') / 255.0
        face_in   = np.expand_dims(face_in, 0)

        preds      = model.predict(face_in, verbose=0)[0]
        top3_idx   = preds.argsort()[-3:][::-1]
        top_name   = le.classes_[top3_idx[0]]
        top_conf   = preds[top3_idx[0]] * 100

        print(f'\nFace {i+1}:')
        for rank, idx in enumerate(top3_idx):
            print(f'  #{rank+1}  {le.classes_[idx]:<20}  {preds[idx]*100:.1f}%')

        label_text = f'{top_name} ({top_conf:.1f}%)'
        color = (0, 200, 0)
        cv2.rectangle(result, (x1, y1), (x2, y2), color, 3)
        cv2.rectangle(result, (x1, y1 - 32), (x2, y1), color, -1)
        cv2.putText(result, label_text, (x1 + 4, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    plt.figure(figsize=(10, 8))
    plt.imshow(result)
    plt.axis('off')
    plt.title('Person Identification Result')
    plt.show()

In [ ]:
image_path = input('Enter full path to image: ')
identify_person(image_path)